# Analyse du modèle GIRT (GKT + IRT)

Évaluation en validation croisée (5 folds) à partir des checkpoints entraînés
(`models/girt/folds/`), avec les mêmes métriques que les autres modèles
(AUC, RMSE, LogLoss) et des analyses spécifiques au modèle : courbes
d'apprentissage, graphe de concepts appris,
PCA des embeddings, paramètres IRT, AUC par
compétence, étude d'ablation du graphe et influence du voisinage.

Le code s'exécute depuis la racine du dépôt comme depuis ce dossier (la racine
est détectée automatiquement). Les anciennes invites interactives (`input()`)
sont remplacées par les variables de configuration ci-dessous.

In [ ]:
import os
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, auc,
    log_loss,
)
from sklearn.decomposition import PCA

# Detect the repository root whether the notebook is launched from the root
# or from models/girt/analysis/.
ROOT = Path.cwd()
while not (ROOT / "datas_management").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "models" / "girt"))
from girt import GIRT, predict, _predict_one_batch
from load_data import load_data, student_kfold_split, build_student_sequences

ANALYSIS_DIR = ROOT / "models" / "girt" / "analysis"
FIGURES_DIR  = ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FOLDS_DIR    = ROOT / "models" / "girt" / "folds"

# ---- Analysis configuration (replaces the original interactive prompts) ----
ANALYSE_FOLD     = None    # 1-based fold number to analyse; None = last available
EXPORT_GRAPH_CSV = False   # True = also export the learned graph as a CSV edge list

## Sauvegarde des figures et métriques

In [ ]:
def _save_fig(name, fig=None):
    """Save the current figure as PDF in ./figures/."""
    path = os.path.join(FIGURES_DIR, f"{name}.pdf")
    if fig is None:
        plt.savefig(path, bbox_inches="tight")
    else:
        fig.savefig(path, bbox_inches="tight")
    return path


def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {
        "AUC":     roc_auc_score(y_true, y_pred),
        "RMSE":    rmse,
        "LogLoss": log_loss(y_true, y_pred),
    }


def expected_calibration_error(y_true, y_pred, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_pred >= bins[i]) & (y_pred < bins[i + 1])
        if mask.sum() > 0:
            ece += np.abs(y_true[mask].mean() - y_pred[mask].mean()) * mask.sum() / len(y_true)
    return ece

## Enregistrement des trajectoires et des portes du GRU

In [ ]:
def record_trajectories(model, tensors, max_students=100, batch_size=64):
    """
    Replay the forward pass and record, for each (student, timestep):
      - skill index
      - predicted probability P(correct)
      - ground truth response
      - L2 norm of H[:, k] (knowledge state intensity for that skill)

    Returns
    -------
    records : list of dicts, one per (student, timestep) observation:
        {"student": int, "t": int, "skill": int, "p_hat": float,
         "y": float, "h_norm": float}
    """
    model.eval()
    seq_item, seq_skill, seq_y, _, seq_mask, _ = tensors
    n_students = min(seq_item.shape[0], max_students)

    seq_item  = seq_item[ :n_students]
    seq_skill = seq_skill[:n_students]
    seq_y     = seq_y[    :n_students]
    seq_mask  = seq_mask[ :n_students]

    # Trim to actual max length
    T_max = int(seq_mask.sum(dim=1).max().item())
    seq_item  = seq_item[ :, :T_max]
    seq_skill = seq_skill[:, :T_max]
    seq_y     = seq_y[    :, :T_max]
    seq_mask  = seq_mask[ :, :T_max]

    records = []

    with torch.no_grad():
        A = model.graph_construction()

        for batch_start in range(0, n_students, batch_size):
            batch_end = min(batch_start + batch_size, n_students)
            B = batch_end - batch_start

            si = seq_item[ batch_start:batch_end]
            sk = seq_skill[batch_start:batch_end]
            sy = seq_y[    batch_start:batch_end]
            sm = seq_mask[ batch_start:batch_end]

            lens    = sm.sum(dim=1)
            T_batch = int(lens.max().item())
            if T_batch == 0:
                continue

            si = si[:, :T_batch]; sk = sk[:, :T_batch]
            sy = sy[:, :T_batch]; sm = sm[:, :T_batch]

            H     = torch.zeros(B, model.d, model.n_skills)
            b_idx = torch.arange(B)

            for t in range(T_batch):
                mask_t = sm[:, t]
                if not mask_t.any():
                    break

                k_t = sk[:, t]
                j_t = si[:, t]
                y_t = sy[:, t]
                logits = model._predict_logits(H, k_t, j_t, A, b_idx)
                preds  = torch.sigmoid(logits)
                h_q    = H[b_idx, :, k_t]

                h_norms = H[b_idx, :, k_t].norm(dim=1)  # (B,) — ||H[:, k]|| per student

                for b in range(B):
                    if mask_t[b]:
                        records.append({
                            "student": batch_start + b,
                            "t":       t,
                            "skill":   k_t[b].item(),
                            "p_hat":   preds[b].item(),
                            "y":       y_t[b].item(),
                            "h_norm":  h_norms[b].item(),
                        })

                # GRU update
                if t < T_batch - 1:
                    y_t_long  = y_t.long()
                    idx_inter = k_t + (1 - y_t_long) * model.n_skills
                    x_t       = model.X[:, idx_inter].T
                    h_tilda   = torch.einsum('bdc,bc->bd', H, A[k_t])
                    new_h     = model.gru_step(x_t, h_tilda)

                    old_h    = H[b_idx, :, k_t]
                    mask_f   = mask_t.float().unsqueeze(1)
                    write_h  = mask_f * new_h + (1 - mask_f) * old_h

                    idx_sc = k_t.view(B, 1, 1).expand(-1, model.d, 1)
                    H = H.scatter(2, idx_sc, write_h.unsqueeze(2))

    return records


def record_gru_gates(model, tensors, batch_size=64):
    """
    Replay the forward pass and record mean reset gate (r_t) and
    update gate (z_t) values for each skill index.

    Returns
    -------
    skill_r : (n_skills,)  mean reset gate per skill
    skill_z : (n_skills,)  mean update gate per skill
    skill_n : (n_skills,)  observation count per skill
    """
    model.eval()
    seq_item, seq_skill, seq_y, _, seq_mask, _ = tensors
    n_students = seq_item.shape[0]

    T_max = int(seq_mask.sum(dim=1).max().item())
    seq_item  = seq_item[ :, :T_max]
    seq_skill = seq_skill[:, :T_max]
    seq_y     = seq_y[    :, :T_max]
    seq_mask  = seq_mask[ :, :T_max]

    r_sum = torch.zeros(model.n_skills)
    z_sum = torch.zeros(model.n_skills)
    n     = torch.zeros(model.n_skills)

    with torch.no_grad():
        A = model.graph_construction()

        for batch_start in range(0, n_students, batch_size):
            batch_end = min(batch_start + batch_size, n_students)
            B = batch_end - batch_start

            si = seq_item[ batch_start:batch_end]
            sk = seq_skill[batch_start:batch_end]
            sy = seq_y[    batch_start:batch_end]
            sm = seq_mask[ batch_start:batch_end]

            lens    = sm.sum(dim=1)
            T_batch = int(lens.max().item())
            if T_batch == 0:
                continue
            si = si[:, :T_batch]; sk = sk[:, :T_batch]
            sy = sy[:, :T_batch]; sm = sm[:, :T_batch]

            H     = torch.zeros(B, model.d, model.n_skills)
            b_idx = torch.arange(B)

            for t in range(T_batch - 1):
                mask_t = sm[:, t]
                if not mask_t.any():
                    break

                k_t      = sk[:, t]
                y_t      = sy[:, t]
                y_t_long = y_t.long()
                idx_inter = k_t + (1 - y_t_long) * model.n_skills
                x_t      = model.X[:, idx_inter].T
                h_tilda  = torch.einsum('bdc,bc->bd', H, A[k_t])

                r = torch.sigmoid(x_t @ model.Wr.T + h_tilda @ model.Ur.T + model.br)
                z = torch.sigmoid(x_t @ model.Wz.T + h_tilda @ model.Uz.T + model.bz)
                h_hat = torch.tanh( x_t @ model.Wh.T + r * (h_tilda @ model.Uh.T) + model.bh)
                new_h = (1 - z) * h_tilda + z * h_hat

                # Accumulate gate means per skill
                for b in range(B):
                    if mask_t[b]:
                        k = k_t[b].item()
                        r_sum[k] += r[b].mean().item()
                        z_sum[k] += z[b].mean().item()
                        n[k]     += 1

                old_h  = H[b_idx, :, k_t]
                mask_f = mask_t.float().unsqueeze(1)
                write_h = mask_f * new_h + (1 - mask_f) * old_h
                idx_sc  = k_t.view(B, 1, 1).expand(-1, model.d, 1)
                H = H.scatter(2, idx_sc, write_h.unsqueeze(2))

    safe_n = n.clamp(min=1)
    return (r_sum / safe_n).numpy(), (z_sum / safe_n).numpy(), n.numpy()

## Courbes ROC, précision-rappel et calibration

In [ ]:
def plot_roc(y_true, y_pred, fold_idx=None):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc_val = roc_auc_score(y_true, y_pred)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc_val:.4f}")
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    title = f"ROC Curve" + (f" — Fold {fold_idx+1}" if fold_idx is not None else "")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    _save_fig("roc")
    plt.show()


def plot_pr(y_true, y_pred, fold_idx=None):
    prec, rec, _ = precision_recall_curve(y_true, y_pred)
    pr_auc = auc(rec, prec)
    plt.figure()
    plt.plot(rec, prec, label=f"PR-AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    title = f"Precision-Recall Curve" + (f" — Fold {fold_idx+1}" if fold_idx is not None else "")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    _save_fig("precision_recall")
    plt.show()


def plot_calibration(y_true, y_pred, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    centers, accs = [], []
    for i in range(n_bins):
        mask = (y_pred >= bins[i]) & (y_pred < bins[i + 1])
        if mask.sum() > 0:
            centers.append(y_pred[mask].mean())
            accs.append(y_true[mask].mean())
    ece = expected_calibration_error(y_true, y_pred, n_bins=n_bins)
    plt.figure()
    plt.plot(centers, accs, marker='o', label=f"ECE = {ece:.4f}")
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label="Perfect calibration")
    plt.xlabel("Predicted probability")
    plt.ylabel("Empirical frequency")
    plt.title("Calibration curve")
    plt.legend()
    plt.tight_layout()
    _save_fig("calibration")
    plt.show()

## Courbes d'apprentissage et normes de l'état de connaissance

In [ ]:
def plot_learning_curves(records, max_students=20, min_obs=10):
    """
    P(correct) over time for each student.
    Uses a smoothed rolling average (window=5) to reduce noise.
    One thin line per student — the overall average in bold.
    """
    from collections import defaultdict
    student_curves = defaultdict(list)
    for rec in records:
        student_curves[rec["student"]].append((rec["t"], rec["p_hat"]))

    students = [s for s, v in student_curves.items() if len(v) >= min_obs][:max_students]
    if not students:
        print("Not enough observations to plot learning curves.")
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    all_smoothed = []
    max_t = 0
    for s in students:
        pairs = sorted(student_curves[s], key=lambda x: x[0])
        ts    = [p[0] for p in pairs]
        ps    = [p[1] for p in pairs]
        # Rolling mean window=5
        smoothed = np.convolve(ps, np.ones(5) / 5, mode='valid')
        ax.plot(smoothed, alpha=0.25, linewidth=0.8, color='steelblue')
        all_smoothed.append(smoothed)
        max_t = max(max_t, len(smoothed))

    # Pad and compute mean trajectory
    padded = np.array([
        np.pad(s, (0, max_t - len(s)), constant_values=np.nan)
        for s in all_smoothed
    ])
    mean_curve = np.nanmean(padded, axis=0)
    ax.plot(mean_curve, color='steelblue', linewidth=2.0, label=f"Mean (n={len(students)} students)")

    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label="p = 0.5 (chance)")
    ax.set_xlabel("Interaction index (chronological, per student)")
    ax.set_ylabel("Predicted P(correct)  [smoothed, window=5]")
    ax.set_title("Student learning curves\n"
                 "Rising = model expects improvement, Flat = no progress detected")
    ax.legend(fontsize='small')
    plt.tight_layout()
    _save_fig("learning_curves")
    plt.show()

## Graphe de concepts appris

In [ ]:
def plot_concept_graph(model, skill_names=None, top_k=None):
    """
    Heatmap of the learned concept adjacency matrix A = softmax(E.T @ E / sqrt(d)).
    A[i, j] is how much skill i attends to skill j's knowledge state during aggregation.

    skill_names : optional list of str of length n_skills
    top_k       : if set, only show the top_k skills by total attention received
    """
    with torch.no_grad():
        A = model.graph_construction().cpu().numpy()  # (C, C) avg heads

    n = A.shape[0]
    if top_k is not None and top_k < n:
        # Keep the top_k skills by total attention received (column sum)
        importance = A.sum(axis=0)
        idx = np.argsort(importance)[::-1][:top_k]
        A = A[np.ix_(idx, idx)]
        labels = [skill_names[i] if skill_names else str(idx[i]) for i in range(len(idx))]
    else:
        labels = skill_names if skill_names else [str(i) for i in range(n)]

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.4 + 2),
                                     max(5, len(labels) * 0.4 + 1)))
    im = ax.imshow(A, aspect='auto', cmap='viridis', vmin=0)
    plt.colorbar(im, ax=ax, label='Attention weight')

    if len(labels) <= 40:
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)

    ax.set_xlabel("Neighbor skill j  (columns = 'broadcasted to')")
    ax.set_ylabel("Current skill i   (rows = 'aggregates from')")
    ax.set_title("Learned concept graph A\n"
                 "A[i,j] = how much skill i attends to skill j during GRU neighbor aggregation")
    plt.tight_layout()
    _save_fig("concept_graph_heatmap")
    plt.show()


def print_top_skill_connections(model, skill_names=None, top_n=5):
    """
    For each skill, print the top_n most attended neighbor skills.
    Useful when n_skills is too large for a readable heatmap.
    """
    with torch.no_grad():
        A = model.graph_construction().cpu().numpy()

    n = A.shape[0]
    name = lambda i: skill_names[i] if skill_names else str(i)
    print(f"\nTop-{top_n} neighbor skills per concept  (averaged over attention heads):")
    print("-" * 60)
    for i in range(n):
        neighbors = np.argsort(A[i])[::-1][:top_n]
        weights   = A[i][neighbors]
        print(f"  Skill {name(i):>20s} → " +
              ", ".join(f"{name(j)} ({w:.3f})" for j, w in zip(neighbors, weights)))


def export_graph_csv(model, skill_names=None, out_path="learned_graph.csv",
                     top_n_per_skill=None, min_weight=None):
    """
    Export the learned adjacency matrix A as a CSV edge list with skill names.

    Each row is (source_id, source_name, target_id, target_name, weight),
    sorted by source then by descending weight. Useful for downstream semantic
    analysis (e.g. open in Excel, Gephi, or NetworkX for community detection).

    top_n_per_skill : if set, keep only the top N outgoing edges per source
    min_weight      : if set, drop edges below this threshold
    """
    import csv
    with torch.no_grad():
        A = model.graph_construction().cpu().numpy()
    n = A.shape[0]
    name = lambda i: skill_names[i] if skill_names else str(i)

    rows = []
    for i in range(n):
        order = np.argsort(A[i])[::-1]
        if top_n_per_skill is not None:
            order = order[:top_n_per_skill]
        for j in order:
            w = float(A[i, j])
            if min_weight is not None and w < min_weight:
                continue
            if i == j:                 # skip self-loop if any
                continue
            rows.append((i, name(i), int(j), name(j), w))

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["source_id", "source_name", "target_id", "target_name", "weight"])
        writer.writerows(rows)
    print(f"  Exported {len(rows)} edges to {out_path}")

## PCA des embeddings de concepts

In [ ]:
def plot_embedding_pca(model, skill_names=None):
    """
    2D PCA of the concept embeddings E (columns = one vector per skill).
    Skills close together share similar embedding representation → model treats them similarly.
    """
    E = model.E.detach().cpu().numpy().T   # (n_skills, d)
    pca = PCA(n_components=2)
    coords = pca.fit_transform(E)           # (n_skills, 2)
    var = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(coords[:, 0], coords[:, 1], alpha=0.7, s=30)

    n = E.shape[0]
    if skill_names:
        for i, name in enumerate(skill_names):
            ax.annotate(name, coords[i], fontsize=6, alpha=0.8)
    elif n <= 50:
        for i in range(n):
            ax.annotate(str(i), coords[i], fontsize=7, alpha=0.8)

    ax.set_xlabel(f"PC1 ({var[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({var[1]*100:.1f}% variance)")
    ax.set_title("PCA of concept embeddings E\n"
                 "Proximity = model treats these skills as similar\n"
                 "(drives neighbor aggregation in the GRU)")
    plt.tight_layout()
    _save_fig("embedding_pca")
    plt.show()

## Paramètres IRT des items

In [ ]:
def plot_item_params(model):
    """
    Effective scalar discrimination (mean softplus(a[:, j]) over d) and
    difficulty (mean b[:, j] over d) per item.
    """
    with torch.no_grad():
        a_eff = F.softplus(model.a_raw).cpu().numpy()   # (n_items,)
        b_eff = model.b.cpu().numpy()                    # (n_items,)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(a_eff, bins=40, color='steelblue', edgecolor='white')
    axes[0].set_xlabel("Effective discrimination  a̅ⱼ  (always > 0)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"Discrimination distribution\n"
                      f"mean={a_eff.mean():.3f}, std={a_eff.std():.3f}, "
                      f"median={np.median(a_eff):.3f}")

    axes[1].hist(b_eff, bins=40, color='coral', edgecolor='white')
    axes[1].axvline(0, color='gray', linestyle='--')
    axes[1].set_xlabel("Effective difficulty  b̅ⱼ  (>0 = hard, <0 = easy)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Difficulty distribution\n"
                      f"mean={b_eff.mean():.3f}, std={b_eff.std():.3f}, "
                      f"median={np.median(b_eff):.3f}")

    axes[2].scatter(b_eff, a_eff, alpha=0.3, s=10, c='purple')
    axes[2].set_xlabel("Difficulty  b̅ⱼ")
    axes[2].set_ylabel("Discrimination  a̅ⱼ")
    axes[2].set_title("Difficulty vs Discrimination\n"
                      "Items top-right: hard AND highly discriminating")

    plt.suptitle("IRT item parameters (averaged over d dimensions)", y=1.02)
    plt.tight_layout()
    _save_fig("item_params")
    plt.show()

## AUC par compétence

In [ ]:
def plot_per_skill_auc(records, skill_names=None):
    """
    AUC per skill: shows which competencies the model predicts well or poorly.
    Bars are sorted by AUC, color-coded by observation count.
    """
    from collections import defaultdict
    skill_ys   = defaultdict(list)
    skill_phat = defaultdict(list)
    for rec in records:
        skill_ys[  rec["skill"]].append(rec["y"])
        skill_phat[rec["skill"]].append(rec["p_hat"])

    aucs, counts, skill_ids = [], [], []
    for k in sorted(skill_ys.keys()):
        yt  = np.array(skill_ys[k])
        yp  = np.array(skill_phat[k])
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        aucs.append(roc_auc_score(yt, yp))
        counts.append(len(yt))
        skill_ids.append(k)

    if not aucs:
        print("Not enough data for per-skill AUC.")
        return

    order  = np.argsort(aucs)
    aucs   = np.array(aucs)[order]
    counts = np.array(counts)[order]
    ids    = [skill_ids[i] for i in order]
    labels = [skill_names[i] if skill_names else str(i) for i in ids]

    # Color bars by log observation count
    norm  = plt.Normalize(vmin=np.log(counts.min()), vmax=np.log(counts.max()))
    cmap  = cm.get_cmap('YlOrRd')
    colors = cmap(norm(np.log(counts)))

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.35 + 2), 5))
    bars = ax.barh(range(len(aucs)), aucs, color=colors, edgecolor='white')
    ax.axvline(0.5,  color='gray',  linestyle='--', linewidth=0.8, label="AUC = 0.5 (chance)")
    ax.axvline(aucs.mean(), color='navy', linestyle='--', linewidth=1.2,
               label=f"Mean AUC = {aucs.mean():.3f}")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("AUC")
    ax.set_title("Per-skill AUC  (sorted)\n"
                 "Color = observation count  (darker = more data)")
    ax.legend(fontsize='small')

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="log(n observations)")

    plt.tight_layout()
    _save_fig("per_skill_auc")
    plt.show()

    # Print summary
    print(f"\nPer-skill AUC summary ({len(aucs)} skills with ≥10 obs):")
    print(f"  Mean  : {aucs.mean():.4f}")
    print(f"  Std   : {aucs.std():.4f}")
    print(f"  Min   : {aucs.min():.4f}  (skill {ids[0]}{': ' + skill_names[ids[0]] if skill_names else ''})")
    print(f"  Max   : {aucs[-1]:.4f}  (skill {ids[-1]}{': ' + skill_names[ids[-1]] if skill_names else ''})")

## Graphe des compétences (définitions)

Fonctions de disposition (force-directed et PCA) du graphe appris. Les tracés
eux-mêmes sont produits plus bas, dans la section dédiée. Au besoin, augmenter
`percentile` pour n'afficher que les connexions les plus fortes si le graphe
est trop dense.

In [ ]:
def _spring_layout(n, edge_list, weights, n_iter=300, k=None, seed=42,
                   spread=2.5, gravity=0.04):
    """
    Fruchterman-Reingold force-directed layout (pure numpy, no networkx needed).

    edge_list : list of (i, j) tuples
    weights   : parallel list of edge weights in [0, 1]
    spread    : multiplies the optimal inter-node distance k — higher = more spread out
    gravity   : weak pull toward origin — prevents isolated nodes from flying off
    returns   : (n, 2) positions in [-1, 1]
    """
    rng = np.random.default_rng(seed)
    pos = rng.uniform(-1.0, 1.0, (n, 2))
    k   = (k or 1.0 / np.sqrt(max(n, 1))) * spread

    for step in range(n_iter):
        force = np.zeros((n, 2))

        # --- Repulsion (all pairs) ---
        delta = pos[:, None, :] - pos[None, :, :]  # (n, n, 2)
        dist  = np.linalg.norm(delta, axis=2) + 1e-8
        rep   = k ** 2 / dist
        np.fill_diagonal(rep, 0)
        force += (rep[:, :, None] * delta / dist[:, :, None]).sum(axis=1)

        # --- Attraction (edges only) ---
        for (i, j), w in zip(edge_list, weights):
            d  = pos[j] - pos[i]
            dn = np.linalg.norm(d) + 1e-8
            f  = (dn ** 2 / k) * w
            force[i] += f * d / dn
            force[j] -= f * d / dn

        # --- Gravity: weak pull toward origin (keeps isolated nodes in frame) ---
        force -= gravity * pos

        # Cool temperature over time
        temp = max(0.02, 0.5 * (1.0 - step / n_iter) ** 1.5)
        norm = np.linalg.norm(force, axis=1, keepdims=True) + 1e-8
        pos += (force / norm) * np.minimum(norm, temp)

    # Normalize to [-1, 1]
    pos -= pos.min(axis=0)
    rng_max = pos.max(axis=0)
    rng_max[rng_max < 1e-8] = 1.0
    pos = pos / rng_max * 2 - 1
    return pos


def _draw_graph(ax, pos, edge_list, weights, in_degree, n, labels, title):
    """Draw nodes + directed edges on a given axis."""
    w_arr   = np.array(weights) if len(weights) else np.array([0.0])
    w_min   = w_arr.min()
    w_max   = w_arr.max()
    w_range = max(w_max - w_min, 1e-8)

    # Edges
    for (i, j), w in zip(edge_list, weights):
        wn    = (w - w_min) / w_range
        alpha = 0.15 + 0.65 * wn
        lw    = 0.4  + 2.0  * wn
        xi, yi = pos[i]
        xj, yj = pos[j]
        ax.annotate(
            "", xy=(xj, yj), xytext=(xi, yi),
            arrowprops=dict(
                arrowstyle="-|>",
                color="steelblue",
                alpha=alpha,
                lw=lw,
                mutation_scale=8,
            ),
            zorder=1,
        )

    # Nodes — size and color by in-degree
    deg_arr = np.array([in_degree.get(i, 0) for i in range(n)])
    sizes   = 60 + 180 * (deg_arr / max(deg_arr.max(), 1))
    sc = ax.scatter(pos[:, 0], pos[:, 1],
                    s=sizes, c=deg_arr, cmap="YlOrRd",
                    edgecolors="gray", linewidths=0.5, zorder=2)
    plt.colorbar(sc, ax=ax, label="In-degree (hub score)", shrink=0.7)

    # Labels
    if n <= 60:
        for i in range(n):
            ax.annotate(labels[i], pos[i],
                        fontsize=6, ha="center", va="bottom",
                        xytext=(0, 5), textcoords="offset points", zorder=3)

    ax.set_title(title, fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")


def plot_skill_graph_on_pca(model, skill_names=None, percentile=93, spread=2.5, gravity=0.04):
    """
    Two side-by-side views of the learned skill graph, keeping only the
    top (100 - percentile)% strongest connections from A.

    Left  — Force-directed spring layout: shows pure graph topology.
             Skills are repositioned to minimise edge crossings.
    Right — PCA layout: node positions = PCA(E), same edges overlaid.
             Proximity in PCA + edge = the model both embeds AND connects
             the two skills similarly.
             Proximity without edge (or edge without proximity) reveals
             discrepancies worth investigating.

    Parameters
    ----------
    percentile : float
        Edge weight threshold percentile (default 93 → top 7% of edges).
        Increase to show fewer, stronger connections.
    """
    with torch.no_grad():
        A = model.graph_construction().cpu().numpy()   # (C, C) avg heads
    E  = model.E.detach().cpu().numpy().T   # (n_skills, d)
    n  = A.shape[0]
    labels = [skill_names[i] if skill_names else str(i) for i in range(n)]

    # ---- PCA positions (normalized to [-1, 1]) ----
    pca     = PCA(n_components=2)
    pca_pos = pca.fit_transform(E)
    var     = pca.explained_variance_ratio_
    pca_pos -= pca_pos.min(axis=0)
    pca_rng  = pca_pos.max(axis=0); pca_rng[pca_rng < 1e-8] = 1.0
    pca_pos  = pca_pos / pca_rng * 2 - 1

    # ---- Threshold edges ----
    # Auto-lower the percentile if the chosen one yields no edges
    # (can happen with peaked asymmetric A where top values are very concentrated)
    diag_mask = ~np.eye(n, dtype=bool)
    A_vals    = A[diag_mask]
    for attempt_pct in [percentile, 90, 80, 70, 50]:
        thresh     = np.percentile(A_vals, attempt_pct)
        rows, cols = np.where((A > thresh) & diag_mask)
        if len(rows) > 0:
            if attempt_pct != percentile:
                print(f"  Note: percentile={percentile} gave 0 edges, "
                      f"auto-adjusted to {attempt_pct}")
            percentile = attempt_pct
            break
    else:
        print("  Warning: no edges found even at 50th percentile — skipping graph.")
        return

    weights   = A[rows, cols].tolist()
    edge_list = list(zip(rows.tolist(), cols.tolist()))

    in_degree = {}
    for (_, j) in edge_list:
        in_degree[j] = in_degree.get(j, 0) + 1

    print(f"  Skill graph: {len(edge_list)} edges shown "
          f"(threshold A > {thresh:.4f}, top {100 - percentile:.0f}%)")

    # ---- Normalise weights to [0, 1] for layout + drawing ----
    w_arr  = np.array(weights)
    w_norm = (w_arr - w_arr.min()) / max(w_arr.max() - w_arr.min(), 1e-8)

    # ---- Spring layout ----
    print("  Computing spring layout (this may take a few seconds)...")
    spring_pos = _spring_layout(n, edge_list, w_norm.tolist(), n_iter=400, spread=spread, gravity=gravity)

    # ---- Plot 1: force-directed layout (topology only) ----
    fig1, ax1 = plt.subplots(figsize=(10, 9))
    _draw_graph(
        ax1, spring_pos, edge_list, w_norm.tolist(), in_degree, n, labels,
        title=(f"Learned concept graph — force-directed layout (topology only)\n"
               f"{len(edge_list)} edges · top {100 - percentile:.0f}% of A\n"
               f"Node size/color = in-degree · Edge width = strength"),
    )
    plt.tight_layout()
    _save_fig("graph_force_directed", fig=fig1)
    plt.show()

    # ---- Plot 2: PCA-positioned layout (embedding positions) ----
    fig2, ax2 = plt.subplots(figsize=(10, 9))
    _draw_graph(
        ax2, pca_pos, edge_list, w_norm.tolist(), in_degree, n, labels,
        title=(f"Learned concept graph — PCA layout (embedding positions)\n"
               f"PC1 {var[0]*100:.1f}%  +  PC2 {var[1]*100:.1f}% variance\n"
               f"Edge without PCA proximity → graph and embedding disagree"),
    )
    ax2.axis("on")
    ax2.set_xlabel(f"PC1 ({var[0]*100:.1f}%)", fontsize=8)
    ax2.set_ylabel(f"PC2 ({var[1]*100:.1f}%)", fontsize=8)
    plt.tight_layout()
    _save_fig("graph_pca_layout", fig=fig2)
    plt.show()

## Étude d'ablation du graphe et influence du voisinage

In [ ]:
def _predict_with_A(model, tensors, A_override, batch_size=64):
    """
    Same as predict() but uses A_override instead of the learned graph.
    Used internally for ablation comparisons.
    """
    model.eval()
    seq_item, seq_skill, seq_y, _, seq_mask, orig_idx = tensors
    n_students = seq_item.shape[0]

    preds_chunks = []
    origs_chunks = []

    with torch.no_grad():
        A = A_override

        for batch_start in range(0, n_students, batch_size):
            batch_end = min(batch_start + batch_size, n_students)
            si = seq_item[ batch_start:batch_end]
            sk = seq_skill[batch_start:batch_end]
            sy = seq_y[    batch_start:batch_end]
            sm = seq_mask[ batch_start:batch_end]
            so = orig_idx[ batch_start:batch_end]

            lens    = sm.sum(dim=1)
            T_batch = int(lens.max().item())
            if T_batch == 0:
                continue
            si = si[:, :T_batch]; sk = sk[:, :T_batch]
            sy = sy[:, :T_batch]; sm = sm[:, :T_batch]; so = so[:, :T_batch]

            preds, origs = _predict_one_batch(model, si, sk, sy, sm, so, A)
            preds_chunks.append(preds)
            origs_chunks.append(origs)


    if not preds_chunks:
        return torch.empty(0)
    all_preds = torch.cat(preds_chunks)
    all_orig  = torch.cat(origs_chunks)
    return all_preds[torch.argsort(all_orig)].cpu()


def graph_ablation_study(model, tensors, skill_names=None, batch_size=64):
    """
    Compares model performance under three graph configurations:

    1. Learned A   — the full trained model (baseline)
    2. Identity A  — A = I  (each skill only attends to itself, no cross-skill
                             propagation). If metrics don't drop, the neighbour
                             aggregation is irrelevant.
    3. Uniform A   — A[i,j] = 1/C  (equal attention to all skills, destroys
                             the learned structure). If this matches the learned
                             model, the specific structure of A doesn't matter —
                             only the fact that *some* aggregation happens.

    Prints a metric table and plots a bar chart for quick visual comparison.
    Returns a dict of {variant_name: metrics_dict}.
    """
    seq_item, seq_skill, seq_y, _, seq_mask, orig_idx = tensors
    valid_orig = orig_idx[seq_mask]
    order      = torch.argsort(valid_orig)
    y_true     = seq_y[seq_mask][order].numpy()

    C = model.n_skills

    with torch.no_grad():
        A_learned  = model.graph_construction()
    A_identity = torch.eye(C)
    A_uniform  = torch.full((C, C), 1.0 / C)

    variants = {
        "Learned A (full model)": A_learned,
        "Identity A (no graph)":  A_identity,
        "Uniform A (no structure)": A_uniform,
    }

    results = {}
    print("\n" + "=" * 55)
    print("  Graph ablation study")
    print("=" * 55)
    print(f"  {'Variant':<28}  {'AUC':>6}  {'RMSE':>6}  {'LogLoss':>8}")
    print("  " + "-" * 53)

    for name, A_mat in variants.items():
        p = _predict_with_A(model, tensors, A_mat, batch_size=batch_size).numpy()
        m = compute_metrics(y_true, p)
        results[name] = m
        print(f"  {name:<28}  {m['AUC']:>6.4f}  {m['RMSE']:>6.4f}  {m['LogLoss']:>8.4f}")

    print("=" * 55)

    # Delta vs learned A
    ref = results["Learned A (full model)"]
    print("\n  Delta vs learned A (positive = learned is better):")
    for name, m in results.items():
        if name == "Learned A (full model)":
            continue
        d_auc = ref["AUC"]     - m["AUC"]
        d_rms = m["RMSE"]      - ref["RMSE"]
        d_ll  = m["LogLoss"]   - ref["LogLoss"]
        print(f"  {name:<28}  ΔAUC={d_auc:+.4f}  ΔRMSE={d_rms:+.4f}  ΔLogLoss={d_ll:+.4f}")

    # ── Bar chart ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    names     = list(results.keys())
    short     = ["Learned", "Identity", "Uniform"]
    colors    = ["steelblue", "coral", "mediumseagreen"]

    for ax, metric, better in zip(axes,
                                   ["AUC", "RMSE", "LogLoss"],
                                   ["↑ higher better", "↓ lower better", "↓ lower better"]):
        vals = [results[n][metric] for n in names]
        bars = ax.bar(short, vals, color=colors, edgecolor="white", width=0.5)
        ax.set_title(f"{metric}  ({better})", fontsize=9)
        ax.set_ylim(min(vals) * 0.97, max(vals) * 1.03)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                    f"{v:.4f}", ha="center", va="bottom", fontsize=8)

    plt.suptitle(
        "Graph ablation study\nDoes replacing the learned A with Identity or Uniform change the results?\nSmall \u0394 \u2192 the graph structure matters little.   Large \u0394 \u2192 the graph is essential.",
        fontsize=9,
    )
    plt.tight_layout()
    _save_fig("graph_ablation")
    plt.show()
    return results


def measure_neighbor_influence(model, tensors, batch_size=64, skill_names=None):
    """
    For every interaction, computes the cosine similarity between:
      h_tilda[b]  — the GRU input after neighbor aggregation  (B, d)
      H[b, :, k]  — the skill's own state before aggregation  (B, d)

    Cosine sim = 1  → neighbours add nothing new (h_tilda ≈ H[:, k])
    Cosine sim < 1  → neighbours genuinely alter the aggregated representation

    Reports:
    - Global average cosine similarity (headline number)
    - Per-skill average → which skills benefit most from their neighbours
    - Distribution histogram
    """
    model.eval()
    seq_item, seq_skill, seq_y, _, seq_mask, _ = tensors
    n_students = seq_item.shape[0]

    skill_sim_sum = torch.zeros(model.n_skills)
    skill_sim_n   = torch.zeros(model.n_skills)
    all_sims      = []

    with torch.no_grad():
        A = model.graph_construction()

        for batch_start in range(0, n_students, batch_size):
            batch_end = min(batch_start + batch_size, n_students)
            B = batch_end - batch_start
            si = seq_item[ batch_start:batch_end]
            sk = seq_skill[batch_start:batch_end]
            sy = seq_y[    batch_start:batch_end]
            sm = seq_mask[ batch_start:batch_end]

            lens    = sm.sum(dim=1)
            T_batch = int(lens.max().item())
            if T_batch == 0:
                continue
            si = si[:, :T_batch]; sk = sk[:, :T_batch]
            sy = sy[:, :T_batch]; sm = sm[:, :T_batch]

            H     = torch.zeros(B, model.d, model.n_skills)
            b_idx = torch.arange(B)

            for t in range(T_batch - 1):
                mask_t = sm[:, t]
                if not mask_t.any():
                    break

                k_t    = sk[:, t]
                h_self = H[b_idx, :, k_t]                        # (B, d)  own state
                h_tilda = torch.einsum('bdc,bc->bd', H, A[k_t])  # (B, d)  aggregated

                # Cosine similarity per sample
                sim = F.cosine_similarity(h_tilda, h_self, dim=1)  # (B,)

                for b in range(B):
                    if mask_t[b]:
                        k = k_t[b].item()
                        s = sim[b].item()
                        skill_sim_sum[k] += s
                        skill_sim_n[k]   += 1
                        all_sims.append(s)

                # GRU update (to keep H current)
                y_t       = sy[:, t]
                idx_inter = k_t + (1 - y_t.long()) * model.n_skills
                x_t       = model.X[:, idx_inter].T
                new_h     = model.gru_step(x_t, h_tilda)
                old_h     = h_self
                mask_f    = mask_t.float().unsqueeze(1)
                write_h   = mask_f * new_h + (1 - mask_f) * old_h
                idx_sc    = k_t.view(B, 1, 1).expand(-1, model.d, 1)
                H = H.scatter(2, idx_sc, write_h.unsqueeze(2))

    all_sims = np.array(all_sims)
    safe_n   = skill_sim_n.clamp(min=1)
    per_skill_sim = (skill_sim_sum / safe_n).numpy()

    global_mean = all_sims.mean()
    print(f"\n  Global mean cosine similarity (h_tilda vs H[:, k]): {global_mean:.4f}")
    print(f"  (1.0 = graph adds nothing,  <1 = neighbours alter the representation)")

    # Skills that benefit most from neighbours (lowest self-similarity)
    valid_skills = np.where(skill_sim_n.numpy() >= 10)[0]
    if len(valid_skills):
        order  = np.argsort(per_skill_sim[valid_skills])
        bottom = valid_skills[order[:5]]
        top    = valid_skills[order[-5:]]
        name   = lambda i: skill_names[i] if skill_names else str(i)
        print("\n  Skills most influenced by neighbours (lowest cosine sim):")
        for i in bottom:
            print(f"    skill {name(i):>20s}  sim={per_skill_sim[i]:.4f}  "
                  f"(n={int(skill_sim_n[i])})")
        print("  Skills least influenced (closest to self-only):")
        for i in top[::-1]:
            print(f"    skill {name(i):>20s}  sim={per_skill_sim[i]:.4f}  "
                  f"(n={int(skill_sim_n[i])})")

    # ── Plots ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Distribution
    axes[0].hist(all_sims, bins=50, color="steelblue", edgecolor="white")
    axes[0].axvline(global_mean, color="navy", linestyle="--",
                    label=f"mean = {global_mean:.4f}")
    axes[0].axvline(1.0, color="gray", linestyle=":", linewidth=0.8,
                    label="sim = 1  (no influence)")
    axes[0].set_xlabel("Cosine similarity  h̃ₜ  vs  H[:, k]")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Distribution of neighbour influence\nLeft = neighbours change the aggregation a lot\nRight = neighbours barely change anything")
    axes[0].legend(fontsize="small")

    # Per-skill bar (sorted)
    if len(valid_skills) > 0:
        order_full = np.argsort(per_skill_sim[valid_skills])
        sorted_ids = valid_skills[order_full]
        sorted_sim = per_skill_sim[sorted_ids]
        labels_sk  = [skill_names[i] if skill_names else str(i) for i in sorted_ids]
        colors_sk  = ["coral" if s < global_mean else "steelblue" for s in sorted_sim]
        axes[1].barh(range(len(sorted_sim)), sorted_sim,
                     color=colors_sk, edgecolor="white")
        axes[1].axvline(global_mean, color="navy", linestyle="--",
                        linewidth=1.2, label=f"mean = {global_mean:.4f}")
        axes[1].axvline(1.0, color="gray", linestyle=":", linewidth=0.8)
        if len(sorted_ids) <= 60:
            axes[1].set_yticks(range(len(labels_sk)))
            axes[1].set_yticklabels(labels_sk, fontsize=7)
        axes[1].set_xlabel("Mean cosine similarity  h̃ₜ  vs  H[:, k]")
        axes[1].set_title("Per-skill neighbour influence\nRed = graph changes this skill's state a lot\nBlue = skill mostly relies on its own state")
        axes[1].legend(fontsize="small")

    plt.suptitle(
        "How much do neighbours actually influence each skill's GRU update?\nCosine similarity between aggregated state and the skill's own H[:, k]",
        fontsize=9,
    )
    plt.tight_layout()
    _save_fig("neighbor_influence")
    plt.show()

    return per_skill_sim, all_sims

## Chargement des données et des noms de compétences

`name_from_id` (dans `datas_management/`) traduit les identifiants
`competence_n` en intitulés lisibles ; à défaut, on retombe sur les indices
numériques.

In [ ]:
df = load_data()
folds    = student_kfold_split(df, k=5, seed=42)
n_items  = int(df["item"].max()  + 1)
n_skills = int(df["skill"].max() + 1)

skill_names = None
dm_dir = ROOT / "datas_management"
if dm_dir.is_dir():
    sys.path.append(str(dm_dir))
try:
    import name_from_id
    name_from_id.init()
    skill_names = [
        name_from_id.get_competence_title(f"competence_{i}")
        for i in range(n_skills)
    ]
    n_missing = sum(1 for s in skill_names if s.startswith("[ERROR"))
    if n_missing > 0:
        print(f"Warning: {n_missing}/{n_skills} skill names could not be resolved.")
    else:
        print(f"Loaded {n_skills} skill names from name_from_id.")
except (ImportError, FileNotFoundError) as e:
    print(f"name_from_id unavailable ({type(e).__name__}: {e}) - falling back to numeric IDs.")
    skill_names = None

## Évaluation en validation croisée (5 folds)

In [ ]:
all_metrics = []

# Cache every successfully loaded fold so a fold can be picked later
# fold_idx -> dict(model, test_tensors, y_true, p)
loaded_folds = {}

for fold_idx, (train_df, test_df) in enumerate(folds):
    print(f"\n--- Fold {fold_idx + 1} ---")

    ckpt_path = FOLDS_DIR / f"girt_fold{fold_idx}.pth"
    try:
        checkpoint = torch.load(ckpt_path, map_location="cpu")
    except FileNotFoundError:
        print(f"  Model checkpoint for fold {fold_idx} not found. Skipping.")
        continue

    test_tensors = build_student_sequences(test_df)

    model = GIRT(
        n_skills=checkpoint["n_skills"],
        n_items=checkpoint["n_items"],
        d=checkpoint["d"],
    )
    model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    model.eval()

    with torch.no_grad():
        gate   = torch.sigmoid(model.alpha_gate).item()
        a_eff  = F.softplus(model.a_raw)
        a_mean = a_eff.mean().item()
        a_std  = a_eff.std().item()
        b_std  = model.b.std().item()
    print(f"  alpha_gate (sigmoid) = {gate:.3f}  (0 = ignores graph, 1 = uses graph fully)")
    print(f"  IRT a: mean={a_mean:.3f} std={a_std:.3f}   b: std={b_std:.3f}")

    n_test_students = test_tensors[0].shape[0]
    n_test_obs      = int(test_tensors[4].sum().item())
    print(f"  Evaluating on TEST set: {n_test_students} students, {n_test_obs} observations")

    seq_item_t, seq_skill_t, seq_y_t, _, seq_mask_t, orig_idx_t = test_tensors
    valid_orig = orig_idx_t[seq_mask_t]
    order      = torch.argsort(valid_orig)
    y_true     = seq_y_t[seq_mask_t][order].numpy()

    with torch.no_grad():
        A_cv = model.graph_construction()
    p_raw, p_orig = [], []
    for bs in range(0, n_test_students, 64):
        be = min(bs + 64, n_test_students)
        si = seq_item_t[bs:be];  sk = seq_skill_t[bs:be]
        sy = seq_y_t[bs:be];     sm = seq_mask_t[bs:be];  so = orig_idx_t[bs:be]
        T_b = int(sm.sum(dim=1).max().item())
        if T_b == 0:
            continue
        preds, origs = _predict_one_batch(
            model, si[:, :T_b], sk[:, :T_b], sy[:, :T_b],
            sm[:, :T_b], so[:, :T_b], A_cv
        )
        p_raw.append(preds); p_orig.append(origs)
    p_all = torch.cat(p_raw)
    o_all = torch.cat(p_orig)
    p     = p_all[torch.argsort(o_all)].cpu().detach().numpy()

    if len(p) != len(y_true):
        print(f"  WARNING: len(p)={len(p)} != len(y_true)={len(y_true)} - skipping fold")
        continue

    metrics = compute_metrics(y_true, p)
    ece     = expected_calibration_error(y_true, p)
    metrics["ECE"] = ece
    all_metrics.append(metrics)

    print(f"  AUC:     {metrics['AUC']:.4f}")
    print(f"  RMSE:    {metrics['RMSE']:.4f}")
    print(f"  LogLoss: {metrics['LogLoss']:.4f}")
    print(f"  ECE:     {ece:.4f}")

    loaded_folds[fold_idx] = {
        "model":        model,
        "test_tensors": test_tensors,
        "y_true":       y_true,
        "p":            p,
    }

print("\n" + "=" * 45)
print("  Cross-validation summary")
print("=" * 45)
if all_metrics:
    if len(all_metrics) == 1:
        print("  (Only 1 fold evaluated - std is 0 by definition)")
    for metric in ["AUC", "RMSE", "LogLoss", "ECE"]:
        vals = [m[metric] for m in all_metrics]
        print(f"  {metric:8s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")
else:
    print("  No folds evaluated - train at least one fold first.")

## Tracés standards pour le fold sélectionné

Régler `ANALYSE_FOLD` dans la cellule de configuration en tête de notebook, puis
exécuter. Le graphe des compétences est tracé dans la section suivante.

In [ ]:
if not loaded_folds:
    print("No fold successfully loaded - cannot plot.")
else:
    available = sorted(loaded_folds.keys())
    default_choice = available[-1]

    if ANALYSE_FOLD is not None and (ANALYSE_FOLD - 1) in loaded_folds:
        fold_idx = ANALYSE_FOLD - 1
    else:
        if ANALYSE_FOLD is not None:
            print(f"Fold {ANALYSE_FOLD} not available - defaulting to fold {default_choice + 1}")
        fold_idx = default_choice

    chosen       = loaded_folds[fold_idx]
    model        = chosen["model"]
    test_tensors = chosen["test_tensors"]
    y_true       = chosen["y_true"]
    p            = chosen["p"]
    print(f"\nAnalysing fold {fold_idx + 1}...")

    plot_roc(y_true, p, fold_idx=fold_idx)
    plot_pr(y_true, p, fold_idx=fold_idx)
    plot_calibration(y_true, p)

    print("Recording trajectories (this may take a moment)...")
    records = record_trajectories(model, test_tensors, max_students=200)

    plot_learning_curves(records, max_students=30, min_obs=10)
    plot_per_skill_auc(records, skill_names=skill_names)

    plot_item_params(model)

## Graphe des compétences appris (force-directed + PCA)

Trace les **deux** vues du graphe appris : disposition force-directed (topologie
pure) puis disposition PCA (positions = PCA des embeddings, mêmes arêtes
superposées). Suivent l'étude d'ablation du graphe et la mesure d'influence du
voisinage. Mettre `EXPORT_GRAPH_CSV = True` dans la configuration pour exporter
aussi le graphe en CSV.

In [ ]:
if loaded_folds:
    # Two views of the learned skill graph: force-directed (topology) and PCA.
    plot_skill_graph_on_pca(model, skill_names=skill_names,
                            percentile=93, spread=3.5, gravity=0.2)

    if n_skills > 15:
        print_top_skill_connections(model, skill_names=skill_names, top_n=5)

    if EXPORT_GRAPH_CSV:
        export_graph_csv(
            model,
            skill_names=skill_names,
            out_path=ANALYSIS_DIR / f"learned_graph_fold{fold_idx + 1}.csv",
            top_n_per_skill=10,
        )

    print("\nRunning graph ablation study...")
    graph_ablation_study(model, test_tensors, skill_names=skill_names)

    print("\nMeasuring neighbour influence...")
    measure_neighbor_influence(model, test_tensors, skill_names=skill_names)

    print("\nDone.")